Por que Dicionários são Essenciais para Engenharia de Dados?

1. Mapeamento e Enriquecimento: É a forma mais rápida de "procurar" um valor. Por exemplo, "Qual é o nome do produto com ID 901?". O dicionário dá essa resposta instantaneamente: produtos[901].

2. Agregação e Agrupamento: Dicionários são a ferramenta perfeita para agrupar dados. Por exemplo, "Qual o total de vendas por categoria?" ou "Quais usuários realizaram quais ações?".

3. Representação de Dados: É a estrutura base do JSON, o formato mais comum para APIs e arquivos de configuração. Um dicionário em Python é um JSON em sua essência.

In [4]:
"""
Enriquecimento de Dados (Data Enrichment)

Conceitos: Acesso d[key], verificação in, e o método .get().

Cenário: Você tem um stream de dados de vendas chegando apenas 
com IDs. Você precisa "enriquecer" esses dados adicionando 
informações legíveis (como nome do produto e categoria) 
antes de carregá-los no Data Warehouse. Um dicionário é perfeito 
para isso, atuando como uma tabela de "lookup" (consulta) em 
memória.
"""
# Dados brutos chegando do stream (lista de tuplas)
vendas_stream = [
    (1, 9001, 150.00),
    (2, 9002, 45.00),
    (3, 9004, 29.90)  # Cuidado: ID 9004 não existe no nosso mapa!
]

#Tabela de "lookup" (poderia vir de um banco de dados)
mapa_produtos = {
    9001: {'nome': 'Teclado Mecanico', 'categoria': 'Perifericos'},
    9002: {'nome': 'Mouse', 'categoria': 'perifericos'},
    9003: {'nome': 'Monitor 20', 'categoria': 'Monitores'}
}

vendas=[] #tupla
for reg in vendas_stream:
    #Desempacotando para melhor entendimento
    id_venda, id_produto, valor = reg

    # Usando .get() para evitar erros (como ensinado no texto)
    # Se o id_produto não for encontrado, ele retorna um 
    #   dicionário padrão.
    inf_produto = mapa_produtos.get(id_produto, {'nome': 'Produto Desconhecido', 'categoria': 'N/A'})

    #criando um novo dicionario com os dados combinados
    registro = {
        'id_venda': id_venda,
        'id_produto': id_produto,
        'produto': inf_produto['nome'],
        'categoria': inf_produto['categoria'],
        'valor': valor
    }
    vendas.append(registro)

import pprint
pprint.pprint(vendas)

[{'categoria': 'Perifericos',
  'id_produto': 9001,
  'id_venda': 1,
  'produto': 'Teclado Mecanico',
  'valor': 150.0},
 {'categoria': 'perifericos',
  'id_produto': 9002,
  'id_venda': 2,
  'produto': 'Mouse',
  'valor': 45.0},
 {'categoria': 'N/A',
  'id_produto': 9004,
  'id_venda': 3,
  'produto': 'Produto Desconhecido',
  'valor': 29.9}]


In [2]:
"""
Agrupando Dados (O Padrão defaultdict)

Conceitos: defaultdict.

Cenário: Você tem uma lista gigante de logs de acesso e precisa
agrupar todas as ações (como 'login', 'logout', 'view_page') 
por usuário.
"""
from collections import defaultdict

logs_brutos = [
    ('user_123', 'login'),
    ('user_456', 'view_page'),
    ('user_123', 'view_page'),
    ('user_123', 'add_to_cart'),
    ('user_456', 'logout'),
    ('user_123', 'logout')
]

#criando um dicinário onde o valor padrão de uma chave é uma
#  lista vazia

atividades = defaultdict(list)
#defaultdict + list = lista vazia

for usuario, atv in logs_brutos:
    """
    Não precisa verificar se a chave existe
    o [defaultdict] garante que atividades[usuario] seja uma 
        lista vazia
    """
    atividades[usuario].append(atv)

print(atividades)

defaultdict(<class 'list'>, {'user_123': ['login', 'view_page', 'add_to_cart', 'logout'], 'user_456': ['view_page', 'logout']})


In [1]:
from collections import defaultdict

status_jobs = [
    'SUCESSO', 'SUCESSO', 'FALHA', 'SUCESSO', 'EM_EXECUCAO', 
    'FALHA', 'SUCESSO', 'FALHA', 'SUCESSO'
]

contagem = defaultdict(int)

for status in status_jobs:
    chave=(status)
    
    contagem[chave] += 1

print(dict(contagem))

{'SUCESSO': 5, 'FALHA': 3, 'EM_EXECUCAO': 1}


In [2]:
vendas = [
    ('P100', 'Sudeste', 100),
    ('P200', 'Nordeste', 50),
    ('P100', 'Sudeste', 150), # Mesmo produto, mesma região
    ('P100', 'Sul', 30),      # Mesmo produto, região diferente
]

#usar defaultdict(int) para realizar soma. O valor padrão é 30
vendas2=defaultdict(int)

for produto, regiao, valor in vendas:
    #criação de uma chave composta usando tupla
    chave=(produto, regiao)
    
    #adicionando o valor ao total da chave
    vendas2[chave] += valor #Funciona igual [d.get(chave, 0) + valor]
    
#Converter [dict] para normal
print(dict(vendas2))

{('P100', 'Sudeste'): 250, ('P200', 'Nordeste'): 50, ('P100', 'Sul'): 30}


In [ ]:
"""
Contagem de Ocorrências (Agregação Simples)

Conceitos: if/else ou .get(key, default_value).

Cenário: Você está analisando logs de um pipeline de dados e precisa contar quantos jobs terminaram com cada
status ('SUCESSO', 'FALHA', 'EM_EXECUCAO').

Sua Tarefa: Crie um dicionário que armazene a contagem de cada status a partir da lista status_jobs.
"""

from collections import defaultdict

status_jobs = [
    'SUCESSO', 'SUCESSO', 'FALHA', 'SUCESSO', 'EM_EXECUCAO',
    'FALHA', 'SUCESSO', 'FALHA', 'SUCESSO'
]

contagem = defaultdict(int)

for status in status_jobs:
    chave=(status)
    contagem[chave] += 1 # substitui 

print(dict(contagem))

In [ ]:
"""
Mesclando Dicionários (Método .update())
Conceitos: .update().

Cenário: Você tem uma configuração padrão para seus pipelines 
(config_padrao). No entanto, para um job específico 
(config_job_especifico), você precisa sobrescrever algumas chaves 
(como retentativas) e adicionar novas (como timeout).

Sua Tarefa: Crie um dicionário config_final que combine os dois, 
dando prioridade aos valores do job específico.
"""
config_padrao = {
    'executor': 'spark',
    'memoria_gb': 8,
    'retentativas': 3
}

config_especifico = {
    'retentativas': 5,  # Sobrescreve o valor padrão
    'timeout_seg': 3600 # Adiciona um novo valor
}

#criar uma copia para não alterar o dicionário original
config_final = config_padrao.copy()

config_padrao['timeout_seg']='3600'
print(config_padrao)



In [1]:
"""
Detecção de Churn (Cancelamento)

Conceitos: diferença (- ou .difference()).

Cenário: Você tem a lista de todos os clientes ativos em Setembro e a lista de todos 
os clientes ativos em Outubro. Você precisa identificar quais clientes "cancelaram" (churn), ou 
seja, quais estavam ativos em Setembro, mas não estão ativos em Outubro.
"""

ativos_setembro = {'user-A', 'user-B', 'user-C', 'user-D', 'user-E'}
ativos_outubro  = {'user-A', 'user-C', 'user-D', 'user-F', 'user-G'}

cancelado= ativos_setembro - ativos_outubro
print(f"Cancelaram em outubro: {cancelado}")

Cancelaram em outubro: {'user-E', 'user-B'}


In [2]:
"""
Detecção de Anomalias (Discrepância)

Conceitos: Diferença Simétrica (^ ou .symmetric_difference()).

Cenário: Você está fazendo uma auditoria de dados entre dois sistemas que deveriam estar idênticos. Você quer encontrar 
todos os IDs que estão em um sistema ou no outro, mas não em ambos. Isso mostra qualquer ID que esteja faltando em um 
lado ou que seja "órfão" no outro.
"""
ids_A = {10, 20, 30, 40, 50}
ids_B = {10, 20, 30, 60, 70}

list=ids_A^ids_B
print(f"IDs que estão faltando ou sobrando: {list}")

IDs que estão faltando ou sobrando: {50, 70, 40, 60}
